# 07 — Final Inference & Submission

**Purpose:** produce 12 recommendations for **every** customer and write the submission.

Built from empty. For the real submission we rebuild candidates/features using **all** data
(basis = the day after the last transaction, so nothing is held out), score with the trained
ranker, take top-12, **pad cold users with popular items**, and map ids back to original strings.

This is the heaviest notebook, so we score customers in **chunks** to stay within RAM.
(In production you'd factor the candidate/feature code into `recsys_utils`; here it's inlined
so each notebook stays runnable on its own.)

In [1]:
from recsys_utils import *
import numpy as np, pandas as pd, pickle
from scipy.sparse import csr_matrix
from implicit.als import AlternatingLeastSquares
import lightgbm as lgb
mlflow = setup_mlflow()

transactions = load_parquet(PROCESSED_DIR / 'transactions_clean.parquet')
customers    = load_parquet(PROCESSED_DIR / 'customers_clean.parquet')
articles     = load_parquet(PROCESSED_DIR / 'articles_clean.parquet')
customer_map = load_parquet(PROCESSED_DIR / 'customer_map.parquet')

with open(MODEL_DIR / 'lgbm_ranker.pkl', 'rb') as f:
    bundle = pickle.load(f)
ranker, feature_cols, cat_cols = bundle['ranker'], bundle['feature_cols'], bundle['cat_cols']

In [2]:
# Submission basis = ALL data (predict the week AFTER the last observed day).
BASIS_END    = transactions['t_dat'].max() + pd.Timedelta(days=1)
REPURCHASE_N = 20
TREND_ADD    = 30
ALS_TOPN     = 100
CHUNK        = 100_000   # customers scored per chunk
META_COLS = ['product_type_name', 'product_group_name', 'colour_group_name',
             'department_name', 'index_group_name', 'garment_group_name']

In [3]:
# ---- Global popularity fallback (last 14 days) ----
pop_win = transactions[transactions['t_dat'] >= BASIS_END - pd.Timedelta(days=14)]
POP12   = pop_win['article_id'].value_counts().head(TOP_K).index.to_numpy()
print('Fallback top-12:', [format_article_id(a) for a in POP12])

Fallback top-12: ['0909370001', '0924243001', '0918522001', '0865799006', '0751471001', '0448509014', '0923758001', '0762846027', '0924243002', '0918292001', '0915529003', '0850917001']


In [4]:
# ---- Train ALS once on all data, keep model + mappings for batch scoring ----
hist = transactions[['customer_id', 'article_id']].drop_duplicates()
u_cat = pd.Categorical(hist['customer_id'])
i_cat = pd.Categorical(hist['article_id'])
ui = csr_matrix((np.ones(len(hist), dtype='float32'), (u_cat.codes, i_cat.codes)),
                shape=(len(u_cat.categories), len(i_cat.categories)))
als = AlternatingLeastSquares(factors=64, regularization=0.01, iterations=20, random_state=RANDOM_SEED)
als.fit(ui)
user_pos  = pd.Series(np.arange(len(u_cat.categories)), index=np.asarray(u_cat.categories))
item_cats = np.asarray(i_cat.categories)
print('ALS trained on full data.')

C:\Users\Michael Fulling\anaconda3\Lib\site-packages\implicit\cpu\als.py:96: RuntimeWarning: Intel MKL BLAS is configured to use 24 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'MKL_NUM_THREADS=1' or by callng 'threadpoolctl.threadpool_limits(1, "blas")'. Having MKL use a threadpool can lead to severe performance issues
  check_blas_config()


  0%|          | 0/20 [00:00<?, ?it/s]

ALS trained on full data.


In [5]:
# ---- Precompute feature tables ONCE on the full basis (reused for every chunk) ----
h = transactions  # basis is all data
u = (h.groupby('customer_id')
      .agg(user_n_tx=('article_id', 'count'), user_avg_spend=('price', 'mean'),
           user_n_distinct=('article_id', 'nunique'), user_last=('t_dat', 'max')).reset_index())
u['user_days_since_last'] = (BASIS_END - u['user_last']).dt.days.astype('int32'); u = u.drop(columns='user_last')
for d in (7, 30, 90):
    c = h[h['t_dat'] >= BASIS_END - pd.Timedelta(days=d)].groupby('customer_id')['article_id'].count().rename(f'user_cnt_{d}d')
    u = u.merge(c, on='customer_id', how='left')

i = (h.groupby('article_id')
      .agg(item_n_tx=('customer_id', 'count'), item_avg_price=('price', 'mean'),
           item_first=('t_dat', 'min')).reset_index())
i['item_days_since_first'] = (BASIS_END - i['item_first']).dt.days.astype('int32'); i = i.drop(columns='item_first')
s7 = h[h['t_dat'] >= BASIS_END - pd.Timedelta(days=7)].groupby('article_id').size().rename('item_sales_7d')
sp = h[(h['t_dat'] >= BASIS_END - pd.Timedelta(days=14)) & (h['t_dat'] < BASIS_END - pd.Timedelta(days=7))].groupby('article_id').size().rename('item_sales_prior7d')
i = i.merge(s7, on='article_id', how='left').merge(sp, on='article_id', how='left')
i[['item_sales_7d', 'item_sales_prior7d']] = i[['item_sales_7d', 'item_sales_prior7d']].fillna(0)
i['item_velocity']  = i['item_sales_7d'] - i['item_sales_prior7d']
i['item_price_pct'] = i['item_avg_price'].rank(pct=True)

ub = h[['customer_id', 'article_id']].drop_duplicates(); ub['ui_bought_before'] = np.int8(1)
meta = articles[['article_id'] + META_COLS]
trend_top = i.sort_values('item_n_tx', ascending=False).head(TREND_ADD)['article_id'].to_numpy()
print('Feature tables ready.')

Feature tables ready.


In [6]:
# Build the same candidate set + features as notebook 05, for a chunk of customers.
def build_chunk(cust_ids):
    cust_ids = np.asarray(cust_ids, dtype='int32')
    # repurchase
    hc = transactions[transactions['customer_id'].isin(cust_ids)].sort_values(['customer_id', 't_dat'])
    rep = (hc.groupby('customer_id', group_keys=False).tail(REPURCHASE_N)[['customer_id', 'article_id']].drop_duplicates())
    rep['source'] = 'repurchase'
    # als (only users present in the ALS matrix)
    su = np.intersect1d(cust_ids, user_pos.index.values)
    if len(su):
        li = user_pos.loc[su].values
        ids, sc = als.recommend(li, ui[li], N=ALS_TOPN, filter_already_liked_items=True)
        als_df = pd.DataFrame({'customer_id': np.repeat(su, ALS_TOPN).astype('int32'),
                               'article_id': item_cats[ids.ravel()].astype('int32'),
                               'als_score': sc.ravel().astype('float32'),
                               'als_rank': np.tile(np.arange(1, ALS_TOPN + 1), len(su)).astype('int16'),
                               'source': 'als'})
    else:
        als_df = pd.DataFrame(columns=['customer_id', 'article_id', 'als_score', 'als_rank', 'source'])
    base = pd.concat([rep, als_df], ignore_index=True).drop_duplicates(['customer_id', 'article_id'])
    # trending expansion
    trend_exp = pd.DataFrame({'customer_id': np.repeat(cust_ids, len(trend_top)).astype('int32'),
                              'article_id': np.tile(trend_top, len(cust_ids)).astype('int32')})
    trend_exp['source'] = 'trending'
    cands = pd.concat([base, trend_exp], ignore_index=True).drop_duplicates(['customer_id', 'article_id']).reset_index(drop=True)
    # features (same joins as nb05)
    df = (cands.merge(u, on='customer_id', how='left').merge(i, on='article_id', how='left')
               .merge(ub, on=['customer_id', 'article_id'], how='left')
               .merge(meta, on='article_id', how='left'))
    df['ui_price_ratio'] = df['item_avg_price'] / (df['user_avg_spend'] + 1e-6)
    df['is_repurchase'] = (df['source'] == 'repurchase').astype('int8')
    df['is_als']        = (df['source'] == 'als').astype('int8')
    df['is_trending']   = (df['source'] == 'trending').astype('int8')
    df['als_score']     = df.get('als_score', pd.Series(0.0, index=df.index)).fillna(0.0).astype('float32')
    df['als_rank']      = df.get('als_rank', pd.Series(999, index=df.index)).fillna(999).astype('int16')
    df['trending_rank'] = 999  # rank not tracked at inference; flag is enough
    df['ui_bought_before'] = df['ui_bought_before'].fillna(0).astype('int8')
    num = df.select_dtypes(include=['float64', 'float32', 'int64', 'int32', 'int16']).columns
    df[num] = df[num].fillna(0)
    for c in META_COLS:
        df[c] = df[c].fillna('Unknown').astype('category')
    return df

In [7]:
# ---- Score all customers in chunks, top-12, pad with popularity ----
all_customers = customers['customer_id'].to_numpy()
preds = {}
for start in range(0, len(all_customers), CHUNK):
    chunk = all_customers[start:start + CHUNK]
    df = build_chunk(chunk)
    if len(df):
        df['score'] = ranker.predict(df[feature_cols])
        top = (df.sort_values(['customer_id', 'score'], ascending=[True, False])
                 .groupby('customer_id')['article_id'].apply(lambda s: s.head(TOP_K).tolist()).to_dict())
    else:
        top = {}
    for cid in chunk:
        rec = top.get(cid, [])
        if len(rec) < TOP_K:                      # pad cold/short users with popular items
            rec = rec + [a for a in POP12 if a not in rec]
        preds[cid] = rec[:TOP_K]
    print(f'scored {min(start + CHUNK, len(all_customers)):,}/{len(all_customers):,}')

scored 100,000/1,371,980
scored 200,000/1,371,980
scored 300,000/1,371,980
scored 400,000/1,371,980
scored 500,000/1,371,980
scored 600,000/1,371,980
scored 700,000/1,371,980
scored 800,000/1,371,980
scored 900,000/1,371,980
scored 1,000,000/1,371,980
scored 1,100,000/1,371,980
scored 1,200,000/1,371,980
scored 1,300,000/1,371,980
scored 1,371,980/1,371,980


In [8]:
# ---- Format submission: original customer_id string + space-joined 10-digit article ids ----
code_to_str = dict(zip(customer_map['customer_id'], customer_map['customer_id_original']))
rows = [(code_to_str[cid], ' '.join(format_article_id(a) for a in recs)) for cid, recs in preds.items()]
submission = pd.DataFrame(rows, columns=['customer_id', 'prediction'])
out_path = OUTPUT_DIR / 'submission.csv'
submission.to_csv(out_path, index=False)
print('Wrote', out_path, submission.shape)
submission.head()

Wrote C:\Users\Michael Fulling\H&M Project\data\outputs\submission.csv (1371980, 2)


,customer_id,prediction
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0568601043 0568601006 0652730004 0850917001 05...
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,0673677002 0714790020 0573085028 0448509014 07...
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0794321007 0852643001 0852643003 0751471001 06...
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,0706016002 0918292001 0706016001 0448509014 07...
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,0818320001 0730683050 0896152002 0791587015 09...


In [9]:
with mlflow.start_run(run_name='07_final_submission'):
    mlflow.log_param('chunk_size', CHUNK)
    mlflow.log_metric('customers_predicted', len(submission))
    mlflow.log_artifact(str(out_path))
print('Logged submission.')

2026/06/11 13:23:42 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet



Logged submission.
